In [2]:
import sys
print(sys.version)

3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]


In [3]:
pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
    --------------------------------------- 2.1/113.8 MB 13.9 MB/s eta 0:00:09
   - -------------------------------------- 3.4/113.8 MB 9.2 MB/s eta 0:00:13
   - -------------------------------------- 5.0/113.8 MB 8.1 MB/s eta 0:00:14
   -- ------------------------------------- 6.3/113.8 MB 7.5 MB/s eta 0:00:15
   -- ------------------------------------- 7.6/113.8 MB 7.3 MB/s eta 0:00:15
   --- ------------------------------------ 8.9/113.8 MB 7.0 MB/s eta 0:00:15
   --- ------------------------------------ 10.2/113.8 MB 6.9 MB/s eta 0:00:16
   ---- ----------------------------------- 11.5/113.8 MB 6.8 MB/s eta 0:00:16
   ---- ----------------------------------- 12.8/113.8 MB 6.7 MB/s eta 0:00:16
   ---- ----------------------------------- 14.2/113.8 MB 6.6 MB/s eta 0:00:16
   ----- ---------------------------------- 15.5/113.8 MB 6.6 MB/s


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Pratham\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [4]:
# Multi Head Attention (from scratch)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()

        self.heads = heads
        self.d_k = d_model // heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x):

        B, S, D = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(B, S, self.heads, self.d_k).transpose(1,2)
        K = K.view(B, S, self.heads, self.d_k).transpose(1,2)
        V = V.view(B, S, self.heads, self.d_k).transpose(1,2)

        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)

        attn = torch.softmax(scores, dim=-1)

        out = torch.matmul(attn, V)

        out = out.transpose(1,2).contiguous().view(B, S, D)

        return self.fc(out)


In [5]:
# Transformer Block
class TransformerBlock(nn.Module):

    def __init__(self, d_model, heads):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, heads)

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        x = self.norm1(x + self.attn(x))

        x = self.norm2(x + self.ff(x))

        return x

In [6]:
# Full Model
class SimpleTransformer(nn.Module):

    def __init__(self, vocab_size, d_model, heads):

        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)

        self.pos = PositionalEncoding(d_model)

        self.block = TransformerBlock(d_model, heads)

        self.fc = nn.Linear(d_model, vocab_size)


    def forward(self, x):

        x = self.embed(x)

        x = self.pos(x)

        x = self.block(x)

        return self.fc(x)

In [7]:
# Test
model = SimpleTransformer(vocab_size=1000, d_model=64, heads=4)

sample = torch.randint(0,1000,(1,5))

out = model(sample)

print(out.shape)

torch.Size([1, 5, 1000])
